# LoTTE RAG — Information Retrieval System Demo
### Comparing TF-IDF, BM25, Dense (all-MiniLM-L6-v2), and Hybrid RRF on LoTTE (Lifestyle / Dev / Search)

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

from src.load import load_lotte, compute_and_print_stats
from src.preprocess import tokenize
from src.sparse import InvertedIndex, TFIDFRetriever, BM25Retriever
from src.dense import encode_corpus, DenseRetriever
from src.fusion import reciprocal_rank_fusion

## 1. Dataset Statistics & Sanity Check
Evaluating on the **LoTTE Lifestyle** domain (Dev split, Search queries).

In [ ]:
passages, queries, qrels = load_lotte()
stats = compute_and_print_stats(passages, queries, qrels)

stats_df = pd.DataFrame([
    {"Metric": "Corpus Size (Passages)", "Value": f"{stats['corpus_size']:,}"},
    {"Metric": "Query Count", "Value": f"{stats['query_count']:,}"},
    {"Metric": "Avg Passage Length (Words)", "Value": f"{stats['avg_passage_length_words']:.2f}"},
    {"Metric": "Relevant Passages / Query", "Value": f"{stats['avg_rel_per_query']:.2f} (min {stats['min_rel_per_query']}, max {stats['max_rel_per_query']})"},
    {"Metric": "Qrels Sanity Check", "Value": "PASSED (100% in corpus)" if stats['sanity_check_passed'] else "FAILED"}
])
display(stats_df)

## 2. Load Retrievers

In [ ]:
from src.run_day1 import get_preprocessed_corpus

# 1. Sparse Index & Models
tokenized_corpus = get_preprocessed_corpus(passages)
inv_idx = InvertedIndex().fit(tokenized_corpus)
tfidf = TFIDFRetriever(inv_idx)
bm25 = BM25Retriever(inv_idx, k1=1.5, b=0.75)

# 2. Dense Embeddings & FAISS Index
embeddings, doc_ids = encode_corpus(passages)
dense = DenseRetriever(embeddings, doc_ids)

print("All four retrievers successfully loaded in memory!")

## 3. Query Comparison: Top-5 Side-by-Side Across All Four Rankers

In [ ]:
demo_qid = 0
demo_query = queries[demo_qid]
gold_pids = qrels[demo_qid]
demo_q_tokens = tokenize(demo_query)

print(f"Query ({demo_qid}): '{demo_query}'")
print(f"Gold Passage IDs: {gold_pids}\n")

res_tfidf = tfidf.search(demo_q_tokens, top_k=5)
res_bm25 = bm25.search(demo_q_tokens, top_k=5)
res_dense = dense.search(demo_query, top_k=5)
res_rrf = reciprocal_rank_fusion([bm25.search(demo_q_tokens, 100), dense.search(demo_query, 100)], k=60, top_k=5)

systems = [
    ("TF-IDF", res_tfidf),
    ("BM25", res_bm25),
    ("Dense (MiniLM)", res_dense),
    ("Hybrid RRF", res_rrf)
]

for name, res in systems:
    print(f"=== {name} Top-5 ===")
    for rank, (pid, score) in enumerate(res, 1):
        is_gold = "[RELEVANT]" if pid in gold_pids else ""
        snippet = passages.get(pid, "")[:120].replace('\n', ' ')
        print(f"  [{rank}] PID {pid} (score: {score:.4f}) {is_gold}\n      \"{snippet}...\"")
    print()

## 4. Benchmark Evaluation Results Table
Comparison on **P@10, Recall@100, MAP, and nDCG@10**.

In [ ]:
with open("results/metrics_table.json", "r") as f:
    metrics_data = json.load(f)

metrics_df = pd.DataFrame(metrics_data)
display(metrics_df.style.highlight_max(subset=["P@10", "Recall@100", "MAP", "nDCG@10"], color="#c7e9c0"))

## 5. Overlaid 11-Point Precision-Recall Curves

In [ ]:
pr_plot_path = Path("results/precision_recall_curves.png")
if pr_plot_path.exists():
    display(Image(filename=str(pr_plot_path)))
else:
    print("Run src/run_day1.py to generate PR curve plot.")

## 6. Qualitative Case Analysis: BM25 vs Dense

In [ ]:
cases_path = Path("results/case_studies.json")
if cases_path.exists():
    with open(cases_path, "r") as f:
        cases = json.load(f)
    for c in cases:
        display(Markdown(f"### {c['type']}: *\"{c['query']}\"* (QID {c['qid']})"))
        display(Markdown(f"- **BM25 nDCG@10**: {c['bm25_ndcg10']} | **Dense nDCG@10**: {c['dense_ndcg10']}"))
        display(Markdown(f"- **Gold PIDs**: {c['gold_pids']}"))
        display(Markdown(f"- **Analysis**: {c['analysis']}"))
        display(Markdown(f"- **BM25 Top Match**: `PID {c['bm25_top3'][0]['pid']}`: {c['bm25_top3'][0]['text']}"))
        display(Markdown(f"- **Dense Top Match**: `PID {c['dense_top3'][0]['pid']}`: {c['dense_top3'][0]['text']}"))
        print("-" * 70)